<a href="https://colab.research.google.com/github/vikassinngh123/AI-ML-Learning/blob/main/06-Deep-Learning/01-PyTorch/02-PyTorch-Computer-Vision/04_intel_image_classification_transfer_learning_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torchvision
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timeit import default_timer as timer

In [2]:
import kagglehub
path = kagglehub.dataset_download("puneet6060/intel-image-classification")

Using Colab cache for faster access to the 'intel-image-classification' dataset.


In [3]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
import os
import torch

BATCH_SIZE=8

print("Folders inside the downloaded dataset:", os.listdir(path))


custom_cnn_train_transform = transforms.Compose([
                                     transforms.Resize(size=(150, 150)), # Resize all landscapes to 150x150
                                     transforms.ToTensor(),              # Convert to PyTorch tensors (values between 0 and 1)
                                     transforms.ColorJitter(brightness=0.1,
                                                           hue=0.1,
                                                           contrast=0.2,
                                                           saturation=0.1),
                                     transforms.RandomRotation(degrees=10),
                                     transforms.RandomHorizontalFlip(p=0.5),
                                     transforms.RandomVerticalFlip(p=0.5),
                                    ])
custom_cnn_test_transform=transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
])


train_dir = os.path.join(path, "seg_train", "seg_train")

test_dir = os.path.join(path, "seg_test", "seg_test")

#Using ImageFolder to map the directories to labels automatically
custom_cnn_train_data = datasets.ImageFolder(
                                  root=train_dir,
                                  transform=custom_cnn_train_transform
                                  )

custom_cnn_test_data = datasets.ImageFolder(
                                 root=test_dir,
                                 transform=custom_cnn_test_transform
                                 )


custom_cnn_train_dataloader = DataLoader(
                              dataset=custom_cnn_train_data,
                              batch_size=BATCH_SIZE,
                              num_workers=2,
                              pin_memory=True,
                              shuffle=True
                              )
custom_cnn_test_dataloader = DataLoader(
                             dataset=custom_cnn_test_data,
                             batch_size=BATCH_SIZE,
                             num_workers=2,
                             pin_memory=True,
                             shuffle=False)

print(f"\nTotal training images: {len(custom_cnn_train_data)}")
print(f"Total test images: {len(custom_cnn_test_data)}")
print(f"Classes: {custom_cnn_train_data.classes}")

Folders inside the downloaded dataset: ['seg_train', 'seg_pred', 'seg_test']

Total training images: 14034
Total test images: 3000
Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [5]:
custom_cnn_train_dataloader,custom_cnn_test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x79f89eb25be0>,
 <torch.utils.data.dataloader.DataLoader at 0x79f89ec8afd0>)

In [6]:
class custom_cnn_model(nn.Module):
  def __init__(self,input_shape,hidden_units,output_shape):
    super().__init__()
    self.cnn_block_1=nn.Sequential(
                                 nn.Conv2d(
                                           in_channels=input_shape,
                                           out_channels=hidden_units,
                                           kernel_size=3,
                                           stride=1,
                                           padding=1
                                           ),
                                 nn.ReLU(),
                                 nn.BatchNorm2d(hidden_units),
                                 nn.Conv2d(
                                           in_channels=hidden_units,
                                           out_channels=hidden_units,
                                           kernel_size=3,
                                           stride=1,
                                           padding=1
                                          ),
                                 nn.ReLU(),
                                 nn.BatchNorm2d(hidden_units),
                                 nn.MaxPool2d(kernel_size=2,
                                              stride=2)
                                 )

    self.cnn_block_2=nn.Sequential(
                                   nn.Conv2d(
                                           in_channels=hidden_units,
                                           out_channels=hidden_units,
                                           kernel_size=3,
                                           stride=1,
                                           padding=1
                                           ),
                                   nn.ReLU(),
                                   nn.BatchNorm2d(hidden_units),
                                   nn.Conv2d(
                                           in_channels=hidden_units,
                                           out_channels=hidden_units,
                                           kernel_size=3,
                                           stride=1,
                                           padding=1
                                           ),
                                   nn.ReLU(),
                                   nn.BatchNorm2d(hidden_units),
                                   nn.MaxPool2d(kernel_size=2,
                                                stride=2)
                                   )

    self.classifer=nn.Sequential(
                                 nn.Flatten(),
                                 nn.Dropout(p=0.3),
                                 nn.Linear(in_features=hidden_units*37*37,
                                           out_features=128),
                                 nn.ReLU(),
                                 nn.Linear(in_features=128,
                                           out_features=output_shape)
                                 )

  def forward(self,x):
      x=self.cnn_block_1(x)
      x=self.cnn_block_2(x)
      x=self.classifer(x)
      return x

custom_cnn_model=custom_cnn_model(
                                input_shape=3,
                                hidden_units=64,
                                output_shape=6
                                ).to(device)
custom_cnn_model

custom_cnn_model(
  (cnn_block_1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (cnn_block_2): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifer): Sequential(
  

In [7]:
!pip -q install torchmetrics
import torchmetrics

loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=custom_cnn_model.parameters(),
                           weight_decay=1e-4,
                           lr=0.001)
accuracy_fn=torchmetrics.Accuracy(task='multiclass',num_classes=len(custom_cnn_train_data.classes)).to(device)

In [8]:
from tqdm.auto import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed(42)

start_time=timer()
ecophs=10

for ecophs in tqdm(range(ecophs)):
  print(f"Epoch: {ecophs+1}\n-------")
  custom_cnn_model.train()
  train_loss,train_acc=0,0
  for batch,(X,y) in enumerate(custom_cnn_train_dataloader):
    X,y=X.to(device),y.to(device)

    y_pred=custom_cnn_model(X)

    loss=loss_fn(y_pred,y)
    train_loss+=loss.item()
    acc=accuracy_fn(y_pred.argmax(dim=1),y)
    train_acc+=acc.item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  avg_trainloss=train_loss/len(custom_cnn_train_dataloader)
  avg_trainacc=train_acc/len(custom_cnn_train_dataloader)
  print(f"\n-----------------------Epoch: {ecophs+1},Avg_Train_loss: {avg_trainloss:.4f}, Avg_Train_acc: {avg_trainacc*100:.4f}----------------------\n")

end_time=timer()
total_time=end_time-start_time
print(f"Total Training Time: {total_time/60:.3f} minutes")

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1
-------

-----------------------Epoch: 1,Avg_Train_loss: 1.5011, Avg_Train_acc: 46.3462----------------------

Epoch: 2
-------

-----------------------Epoch: 2,Avg_Train_loss: 1.0172, Avg_Train_acc: 57.5000----------------------

Epoch: 3
-------

-----------------------Epoch: 3,Avg_Train_loss: 0.9334, Avg_Train_acc: 62.1652----------------------

Epoch: 4
-------

-----------------------Epoch: 4,Avg_Train_loss: 0.8344, Avg_Train_acc: 68.7892----------------------

Epoch: 5
-------

-----------------------Epoch: 5,Avg_Train_loss: 0.7335, Avg_Train_acc: 73.6040----------------------

Epoch: 6
-------

-----------------------Epoch: 6,Avg_Train_loss: 0.6943, Avg_Train_acc: 74.3732----------------------

Epoch: 7
-------

-----------------------Epoch: 7,Avg_Train_loss: 0.6451, Avg_Train_acc: 76.8661----------------------

Epoch: 8
-------

-----------------------Epoch: 8,Avg_Train_loss: 0.6089, Avg_Train_acc: 77.7920----------------------

Epoch: 9
-------

----------------------

In [9]:
test_loss = 0.0
test_acc = 0.0

custom_cnn_model.eval()
with torch.inference_mode():
    for X_test, y_test in custom_cnn_test_dataloader:
        X_test, y_test = X_test.to(device), y_test.to(device)

        test_pred = custom_cnn_model(X_test)

        test_loss += loss_fn(test_pred, y_test).item()
        test_acc += accuracy_fn(test_pred.argmax(dim=1), y_test).item()

    test_loss /= len(custom_cnn_test_dataloader)
    test_acc /= len(custom_cnn_test_dataloader)

print(f"\nFinal Test Loss: {test_loss:.4f} | Final Test Acc: {test_acc * 100:.2f}%")


Final Test Loss: 0.4989 | Final Test Acc: 82.27%


In [15]:
print(f"Custom CNN Model Total_parameters=10.83mil Trainable_parameters=10.83mil Total_time={total_time:.4f} Test Acc={test_acc:.4f}")

Custom CNN Model Total_parameters=10.83mil Trainable_parameters=10.83mil Total_time=1108.6366 Test Acc=0.8227


In [16]:
import torchvision.models as model
resnet18=model.resnet18(pretrained=True).to(device)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]


In [17]:
from torchvision.models import ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
resnet18_transform = weights.transforms()

resnet18_train_data=datasets.ImageFolder(
                                  root=train_dir,
                                  transform=resnet18_transform
                                  )
resnet18_test_data=datasets.ImageFolder(
                                 root=test_dir,
                                 transform=resnet18_transform
                                 )
resnet18_train_dataloader=DataLoader(
                                      dataset=resnet18_train_data,
                                      batch_size=BATCH_SIZE,
                                      num_workers=2,
                                      pin_memory=True,
                                      shuffle=True
                                     )
resnet18_test_dataloader=DataLoader(
                                    dataset=resnet18_test_data,
                                    batch_size=BATCH_SIZE,
                                    num_workers=2,
                                    pin_memory=True,
                                    shuffle=False
                                    )

In [19]:
for param in resnet18.parameters():
  param.requires_grad=False

In [24]:
resnet18.fc=nn.Sequential(
                                nn.Linear(in_features=512,
                                          out_features=256),
                                nn.ReLU(),
                                nn.Dropout(p=0.3),
                                nn.Linear(in_features=256,
                                          out_features=128),
                                nn.ReLU(),
                                nn.Dropout(p=0.3),
                                nn.Linear(in_features=128,
                                          out_features=6)
                                ).to(device)
resnet18

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [26]:
resnet18_loss_fn=nn.CrossEntropyLoss()
resnet18_optimizer=torch.optim.Adam(params=resnet18.parameters(),
                                     weight_decay=1e-4,
                                     lr=0.001)

In [28]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

resnet18_start_time=timer()
ecophs=10

for ecophs in tqdm(range(ecophs)):
  print(f"Epoch: {ecophs+1}\n-------")
  resnet18_train_loss=0
  resnet18_train_acc=0
  resnet18.train()
  for batch,(X,y) in enumerate(resnet18_train_dataloader):
    X,y=X.to(device),y.to(device)

    y_pred=resnet18(X)

    resnet18_loss=resnet18_loss_fn(y_pred,y)
    resnet18_train_loss+=resnet18_loss.item()
    acc=accuracy_fn(y_pred.argmax(dim=1),y)
    resnet18_train_acc+=acc.item()

    resnet18_optimizer.zero_grad()

    resnet18_loss.backward()

    resnet18_optimizer.step()

  avg_resnet18_trainloss=resnet18_train_loss/len(resnet18_train_dataloader)
  avg_resnet18_trainacc=resnet18_train_acc/len(resnet18_train_dataloader)
  print(f"\n-----------------------Epoch: {ecophs+1},Avg_Train_loss: {avg_resnet18_trainloss:.4f}, Avg_Train_acc: {avg_resnet18_trainacc*100:.4f}----------------------\n")

resnet18_end_time=timer()
resnet18_total_time=(resnet18_end_time-resnet18_start_time)/60
print(f"Total Training Time: {resnet18_total_time:.3f} minutes")

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1
-------

-----------------------Epoch: 1,Avg_Train_loss: 0.6374, Avg_Train_acc: 75.7336----------------------

Epoch: 2
-------

-----------------------Epoch: 2,Avg_Train_loss: 0.4972, Avg_Train_acc: 82.3575----------------------

Epoch: 3
-------

-----------------------Epoch: 3,Avg_Train_loss: 0.4786, Avg_Train_acc: 82.5712----------------------

Epoch: 4
-------

-----------------------Epoch: 4,Avg_Train_loss: 0.4634, Avg_Train_acc: 83.2977----------------------

Epoch: 5
-------

-----------------------Epoch: 5,Avg_Train_loss: 0.4460, Avg_Train_acc: 83.7749----------------------

Epoch: 6
-------

-----------------------Epoch: 6,Avg_Train_loss: 0.4370, Avg_Train_acc: 84.2094----------------------

Epoch: 7
-------

-----------------------Epoch: 7,Avg_Train_loss: 0.4380, Avg_Train_acc: 84.1595----------------------

Epoch: 8
-------

-----------------------Epoch: 8,Avg_Train_loss: 0.4319, Avg_Train_acc: 84.8718----------------------

Epoch: 9
-------

----------------------

In [31]:
resnet18_test_loss = 0.0
resnet18_test_acc = 0.0

resnet18.eval()
with torch.inference_mode():
    for X_test, y_test in resnet18_test_dataloader:
        X_test, y_test = X_test.to(device), y_test.to(device)

        test_pred = resnet18(X_test)

        resnet18_test_loss += resnet18_loss_fn(test_pred, y_test).item()
        resnet18_test_acc += accuracy_fn(test_pred.argmax(dim=1), y_test).item()

    resnet18_test_loss = resnet18_test_loss/len(resnet18_test_dataloader)
    resnet18_test_acc = resnet18_test_acc/len(resnet18_test_dataloader)

print(f"\nFinal Test Loss: {resnet18_test_loss:.4f} | Final Test Acc: {resnet18_test_acc * 100:.2f}%")


Final Test Loss: 0.2701 | Final Test Acc: 90.47%


In [32]:
def count_trainable_params(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_params(model):
  return sum(p.numel() for p in model.parameters())


benchmark_data = {
    "Model": ["Custom CNN", "ResNet-18 (Transfer Learning)"],
    "Trainable Params": [
        f"{count_trainable_params(custom_cnn_model):,}",
        f"{count_trainable_params(resnet18):,}",
    ],
    "Total Params": [
        f"{count_total_params(custom_cnn_model):,}",
        f"{count_total_params(resnet18):,}",
    ],
    "Time (mins)": [
        f"{total_time / 60:.2f}",
        f"{resnet18_total_time:.2f}",
    ],
    "Test Acc (%)": [
        f"{test_acc * 100:.2f}%",
        f"{resnet18_test_acc * 100:.2f}%",
    ],
}

# Create and display DataFrame
benchmark_df = pd.DataFrame(benchmark_data)
display(benchmark_df)

,Model,Trainable Params,Total Params,Time (mins),Test Acc (%)
0,Custom CNN,"11,328,838","11,328,838",18.48,82.27%
1,ResNet-18 (Transfer Learning),"164,998","11,341,510",9.21,90.47%


### Important Note
The custom CNN uses the preprocessing from the original baseline experiment, while ResNet18 uses the preprocessing associated with its pretrained ImageNet weights. Therefore, the comparison reflects the practical performance of the two approaches rather than a strictly controlled architecture-only comparison.